# 🗓️ 28일차 스터디 노트북 — 보이어·무어법 (건너뛰는 검색)

**오늘 범위**: 07-3 보이어·무어법 — 뒤에서부터 비교 · 건너뛰기 표(이동량 표) 만들기 · 실습 7-3 `bm_match` · **문자열 검색 3종 시간 복잡도 총정리** · 보충수업 7-2 `ord`/`chr`

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[계산]**

---

## 발상이 정반대로 뒤집힌다

| | 비교 방향 | 핵심 아이디어 |
|---|---|---|
| 브루트 포스 (26일) | 앞 → 뒤 | 그냥 1칸씩 민다 |
| KMP (27일) | 앞 → 뒤 | **이미 본 것**을 재활용 |
| **보이어·무어 (오늘)** | **뒤 → 앞** | **안 본 것**을 건너뛴다 |

교재 315p: **"보이어·무어법은 KMP법보다 더 효율적이어서 실제 문자열 검색에서 널리 사용하는 알고리즘입니다."**
그리고 **"패턴의 끝 문자에서 시작하여 앞쪽을 향해 검사를 수행합니다."**

🔥 **왜 뒤에서부터 비교하면 이득일까?**
패턴 마지막 글자와 마주친 텍스트 글자가 **패턴에 아예 없는 글자**라면? 그 글자를 지나칠 때까지는 **어떤 위치에서도 매칭이 불가능**해. 그러니 **한 번에 패턴 길이만큼 점프**할 수 있어.

→ **n글자를 전부 볼 필요가 없다.** 이게 오늘 배울 **O(n/m)** 의 정체야.

## 오늘의 두 번째 목표

> **"O(mn), O(n), O(n/m)이 각각 왜 그런지 손으로 계산하고 눈으로 확인하기"**

PART 5를 통째로 여기에 썼어. 세 알고리즘의 비교 횟수를 **직접 세고 실측**할 거야.

## 진행 순서
**개념(1~3) → 패턴 이동 손 추적(4~6) → 건너뛰기 표(7~10) → 코드 구현(11~14) → 시간 복잡도(15~18)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
import random, string, time

# 교재 실습 7-3
def bm_match(txt: str, pat: str) -> int:
    """보이어·무어법으로 문자열 검색"""
    skip = [None] * 256                      # 건너뛰기 표

    # 건너뛰기 표 만들기
    for pt in range(256):
        skip[pt] = len(pat)
    for pt in range(len(pat)):
        skip[ord(pat[pt])] = len(pat) - pt - 1

    # 검색하기
    while pt < len(txt):                     # ⚠️ pt 초기화가 없다! (13번)
        pp = len(pat) - 1
        while txt[pt] == pat[pp]:
            if pp == 0:
                return pt
            pt -= 1
            pp -= 1
        pt += skip[ord(txt[pt])] if skip[ord(txt[pt])] > len(pat) - pp \
            else len(pat) - pp
    return -1


def build_bm_skip(pat):
    """건너뛰기 표만 따로 (실험용)"""
    skip = [len(pat)] * 256
    for i in range(len(pat)):
        skip[ord(pat[i])] = len(pat) - i - 1
    return skip


def show_bm_skip(pat, chars="ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    """건너뛰기 표를 교재 [그림 7-10] 형식으로 출력"""
    sk = build_bm_skip(pat)
    half = len(chars) // 2
    for row in (chars[:half], chars[half:]):
        print("  " + "  ".join(row))
        print("  " + "  ".join(str(sk[ord(c)]) for c in row))
    return sk


def align(txt, pat, end_pos):
    """패턴의 '마지막 글자'가 end_pos에 오도록 그린다"""
    start = end_pos - len(pat) + 1
    print(f"  idx  {' '.join(str(i%10) for i in range(len(txt)))}")
    print(f"  txt  {' '.join(txt)}")
    print(f"  pat  {'  '*max(0,start)}{' '.join(pat)}")


# 이전 알고리즘 (비교용)
def bf_match(txt, pat):
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return pt - pp if pp == len(pat) else -1

def build_kmp_skip(pat):
    pt = 1; pp = 0; sk = [0] * (len(pat) + 1); sk[1] = 0
    while pt != len(pat):
        if pat[pt] == pat[pp]: pt += 1; pp += 1; sk[pt] = pp
        elif pp == 0: pt += 1; sk[pt] = pp
        else: pp = sk[pp]
    return sk

def kmp_match(txt, pat):
    sk = build_kmp_skip(pat)
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif pp == 0: pt += 1
        else: pp = sk[pp]
    return pt - pp if pp == len(pat) else -1

def naive(t, p):
    for i in range(len(t) - len(p) + 1):
        if t[i:i+len(p)] == p: return i
    return -1


# ---------- 비교 횟수 측정용 (PART 5에서 사용) ----------
def bf_count(txt, pat):
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return c

def kmp_count(txt, pat):
    sk = build_kmp_skip(pat); pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif pp == 0: pt += 1
        else: pp = sk[pp]
    return c

def bm_count(txt, pat):
    m = len(pat); skip = build_bm_skip(pat)
    pt = m - 1; c = 0
    while pt < len(txt):
        pp = m - 1
        while txt[pt] == pat[pp]:
            c += 1
            if pp == 0: return c
            pt -= 1; pp -= 1
        c += 1
        a = skip[ord(txt[pt])]; b = m - pp
        pt += a if a > b else b
    return c


TXT = 'ABCXDEZCABACABAC'
PAT = 'ABAC'

print("준비 완료 ✅\n")
align(TXT, PAT, len(PAT)-1)
print(f"\n  n = len(txt) = {len(TXT)}, m = len(pat) = {len(PAT)}")
print(f"  검색 결과: {bm_match(TXT, PAT)}  → txt[8:12] = '{TXT[8:12]}'")

---
# 🔁 [Remind] 워밍업 — 26~27일차 되감기

07장 마지막 날이야. 앞의 두 알고리즘을 정확히 기억해야 비교가 돼.

### R-1. 🟢 [설명] 세 알고리즘의 "불일치 처리"

불일치했을 때 각각 뭘 하는지 한 줄로:

| 알고리즘 | 불일치 시 동작 | 코드 |
|---|---|---|
| 브루트 포스 (26일) | ① | `pt = pt - pp + 1; pp = 0` |
| KMP (27일) | ② | `pp = skip[pp]` |
| 보이어·무어 (오늘) | ③ (예측해봐) | ? |

- ①②를 채우고, ③은 **예측만** 해둬.
- 26일차 7번의 불변식 `pt - pp` 를 다시 적어봐. 오늘도 쓰일까?

### R-2. 🟡 [설명] 두 개의 "건너뛰기 표"는 다르다

27일차 KMP에도 `skip` 표가 있었고, 오늘도 `skip` 표가 나와. **이름은 같은데 완전히 다른 물건**이야.

| | KMP의 skip (27일) | 보이어·무어의 skip (오늘) |
|---|---|---|
| 인덱스가 뜻하는 것 | ① **패턴에서 맞춘 글자 수** | ② |
| 값이 뜻하는 것 | ③ **되돌아갈 pp 값** | ④ |
| 배열 크기 | ⑤ `len(pat) + 1` | ⑥ |

- ②④⑥을 채워봐. (코드의 `skip = [None] * 256` 을 보면 힌트가 있어)
- 왜 크기가 **256**일까? 보충수업 7-2의 `ord()` 와 관련 있어.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 왜 뒤에서부터 비교하나 (1~3번)

### 1. 🟢 [설명] 교재 315p — 패턴에 없는 글자를 만나면

```
  idx  0 1 2 3 4 5 6 7 8 9 ...
  txt  A B C X D E Z C A B A C A B A C
  pat  A B A C
```

교재 315p: **"패턴의 마지막 문자 'C'에 주목합니다. 같은 위치에 있는 텍스트의 'X'는 패턴 안에 포함되어 있지 않습니다."**

- 패턴의 마지막 글자는 `txt[____]` 와 마주 봐? (패턴 길이가 4니까)
- 그 자리의 텍스트 글자는 ①____ 이고, 이 글자는 패턴 `ABAC` 안에 있어? ②____
- 교재 ⓑ~ⓓ: **"패턴을 1칸/2칸/3칸 밀어도 텍스트의 문자 'X'와 패턴의 문자가 일치하지 않습니다."**
  → **왜 그럴까?** 한 문장으로 설명해봐. 🔥
- 그래서 교재는 **"비교하는 과정을 생략하고 패턴을 오른쪽으로 한번에 4칸 밀어"** 버려.
- 💡 핵심: 텍스트의 `X`를 **완전히 지나칠 때까지** 어떤 위치에서도 매칭이 불가능해. `X`가 패턴에 없으니까!

*(답을 적은 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
m = len(pat)
print("패턴의 마지막 글자를 txt[3]에 맞춘 상태:")
align(txt, pat, m-1)
print(f"\n  txt[{m-1}] = '{txt[m-1]}' vs pat[{m-1}] = '{pat[m-1]}' → 불일치")
print(f"  '{txt[m-1]}' 가 패턴 '{pat}' 안에 있나? {txt[m-1] in pat}\n")

print("1~3칸 밀어봐도 소용없는 이유:")
for shift in (1, 2, 3):
    end = m - 1 + shift
    start = end - m + 1
    # X는 txt[3]에 있다. 패턴의 어느 글자와 마주보나?
    idx_in_pat = 3 - start
    print(f"  {shift}칸 밀기 → txt[3]='X' 가 pat[{idx_in_pat}]='{pat[idx_in_pat]}' 와 마주봄 → 불일치 ❌")
print(f"  4칸 밀기 → txt[3]='X' 를 완전히 지나침 ✅\n")
align(txt, pat, m-1+4)
print(f"\n  이제 txt[7]='{txt[7]}' vs pat[3]='{pat[3]}' → {'일치! 🎯' if txt[7]==pat[3] else '불일치'}")

### 2. 🟡 [설명] "패턴에 있는 글자"를 만나면

1번은 **패턴에 없는 글자**(`X`) 이야기였어. 그럼 **있는 글자**를 만나면?

교재 316p [그림 7-8]: 패턴의 마지막 문자 `C`가 텍스트 문자 `A`와 일치하지 않는 상황.
> **"그런데 문자 'A'는 패턴의 1번째와 3번째에 포함되어 있습니다. 그래서 ⓑ처럼 뒤쪽에 있는 'A'가 위아래로 겹치도록 패턴을 오른쪽으로 1칸만 밀어냅니다."**

```
pat = A B A C
      0 1 2 3      ← 'A'는 인덱스 0과 2에 있다
```

- 텍스트의 `A`와 패턴의 `A`를 맞추려면, **어느 쪽 `A`** 를 맞춰야 안전할까? 앞쪽(인덱스 0)? 뒤쪽(인덱스 2)?
- 교재 각주: **"이때 ⓓ처럼 패턴의 앞쪽에 있는 'A'가 위아래로 겹치도록 오른쪽으로 한번에 3칸만큼 밀어서는 안 됩니다."**
  → **왜 안 될까?** 🔥 (힌트: 3칸을 밀면 인덱스 2의 `A`가 어디로 가지? 그 사이에 정답이 있을 수도 있잖아)
- 그래서 이동량은 **"패턴에서 그 글자가 마지막으로 나오는 위치"** 기준으로 정해. `A`는 인덱스 2가 마지막이니까 이동량은 `4 - 2 - 1 = 1`.
- 💡 일반화: 글자 `c`가 패턴의 인덱스 `k`에 마지막으로 나온다면 **이동량 = m - k - 1**

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
print(f"pat = {pat}")
print(f"각 글자의 '마지막 출현 위치'와 이동량:\n")
print("  글자 | 출현 위치 | 마지막 k | 이동량 (m-k-1)")
print("  " + "-"*46)
seen = {}
for i, ch in enumerate(pat):
    seen.setdefault(ch, []).append(i)
for ch, poss in seen.items():
    k = poss[-1]
    print(f"   {ch}   | {str(poss):9s} |    {k}     |   {len(pat)-k-1}")
print(f"\n  패턴에 없는 글자 → 이동량 = m = {len(pat)}")

print("\n[앞쪽 A(인덱스 0)에 맞추면 왜 안 되나]")
print("  3칸 밀면 인덱스 2의 A는 텍스트에서 '이미 지나간' 자리로 간다.")
print("  → 그 사이 위치에 정답이 있어도 확인 없이 건너뛰게 된다 ❌")
print("  → 항상 '가장 뒤쪽 출현'을 기준으로 해야 안전 (보수적 선택)")

### 3. 🟢 [설명] 이름 정리

- **보이어·무어법(Boyer-Moore method)**, 줄여서 **BM법**. 교재 315p 각주: <br>
  **"이 알고리즘을 고안한 R. S. 보이어(R. S. Boyer)와 J. S. 무어(J. S. Moore)의 이름을 따서 붙였으며"**
- 27일차 KMP도 세 사람 이름(Knuth-Morris-Pratt)이었지. 알고리즘에 사람 이름이 붙는 경우가 많아.
- 교재 317p 각주 🔥: **"여기에서는 배열 하나를 사용하여 보이어·무어법을 간략하게 나타냈습니다. 원래의 보이어·무어법은 배열 2개를 사용해서 검사합니다."**
  → 우리가 배우는 건 **간이 버전**이야. 이게 나중에 18번(최악 시간 복잡도)에서 중요해져.

**오늘 배우는 규칙의 정식 명칭**: **나쁜 문자 규칙(bad character rule)**
원래 BM은 여기에 **착한 접미사 규칙(good suffix rule)** 이라는 두 번째 표를 더해.

*(읽고 넘어가기)*

---
# ✍️ PART 2 — 패턴 이동 손으로 추적 (4~6번)

> 교재 315~316p [그림 7-5 ~ 7-9]를 직접 그린다.

### 4. 🟢 [손] 교재 그림 7-5 ~ 7-9 완전 재현

```
  idx  0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  txt  A B C X D E Z C A B  A  C  A  B  A  C
  pat  A B A C
```

**표를 채워봐.** (`pt` = 텍스트에서 지금 보고 있는 위치, `pp` = 패턴에서의 위치)

| 라운드 | pt | pp | txt[pt] | pat[pp] | 결과 | 이동량 | 다음 pt |
|---|---|---|---|---|---|---|---|
| 1 | 3 | 3 | X | C | 불일치 | ① | ② |
| 2 | ③ | 3 | ④ | C | ⑤ | | |
| 3 | | | | | | | |
| 4 | | | | | | | |

**힌트**
- 라운드 1: `X`는 패턴에 없으니 이동량 **4** → `pt = 3 + 4 = 7`
- 라운드 2: `txt[7]='C'` vs `pat[3]='C'` → **일치!** 안쪽으로 들어가서 `pt -= 1, pp -= 1`
  - `txt[6]='Z'` vs `pat[2]='A'` → 불일치. `Z`는 패턴에 없으니 이동량은?
  - ⚠️ 여기서 함정: `pt`가 이미 6으로 **되돌아간** 상태야. 어디서부터 밀어야 하지? (12번에서 정확히 다룸)
- 최종적으로 `pt = ____` 에서 매칭 성공, `return ____`

*(끝까지 채운 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
m = len(pat)
skip = build_bm_skip(pat)
pt = m - 1
rnd = 0
print(f"txt = {txt}\npat = {pat}  (m = {m})\n")
while pt < len(txt):
    rnd += 1
    pp = m - 1
    start_pt = pt
    print(f"[라운드 {rnd}] 패턴 끝을 txt[{pt}]에 맞춤")
    align(txt, pat, pt)
    matched = 0
    found = False
    while txt[pt] == pat[pp]:
        matched += 1
        print(f"    txt[{pt}]='{txt[pt]}' == pat[{pp}]='{pat[pp]}' ✅")
        if pp == 0:
            print(f"    → 전부 일치! return {pt}")
            found = True
            break
        pt -= 1; pp -= 1
    if found: break
    print(f"    txt[{pt}]='{txt[pt]}' != pat[{pp}]='{pat[pp]}' ❌  (뒤에서 {matched}글자 맞춤)")
    a = skip[ord(txt[pt])]
    b = m - pp
    mv = a if a > b else b
    print(f"    이동량 = max(skip['{txt[pt]}']={a}, m-pp={b}) = {mv}")
    pt += mv
    print(f"    → pt = {pt}\n")

print(f"\n최종 결과: {bm_match(txt, pat)}")

### 5. 🟡 [손] 다른 예제 — 직접 굴려보기

```
  txt  H E R E _ I S _ A _ S I M P L E _ E X A M P L E
  pat  E X A M P L E
```
(공백은 `_` 로 표시했지만 실제로는 스페이스)

- 먼저 `pat = 'EXAMPLE'` 의 **각 글자 이동량**을 손으로 구해봐. (2번 공식: `m - k - 1`, k는 마지막 출현 위치)

| 글자 | E | X | A | M | P | L | 그 외 |
|---|---|---|---|---|---|---|---|
| 마지막 출현 k | ① | ② | ③ | ④ | ⑤ | ⑥ | - |
| 이동량 | ⑦ | ⑧ | ⑨ | ⑩ | ⑪ | ⑫ | ⑬ |

🔥 **`E`를 주의해.** `EXAMPLE`에서 `E`는 인덱스 0과 6에 나와. 마지막 출현은 6이니 이동량은 `7 - 6 - 1 = 0`.
- 이동량이 **0**이면 패턴이 안 움직인다는 뜻인데, 그럼 무한 루프 아니야? (12번에서 해결)

- 그다음 **몇 라운드** 만에 찾는지 추적해봐. 텍스트 길이가 24인데 비교를 몇 번 할까?

*(예측을 적은 뒤 실행)*

In [ ]:
txt2 = "HERE IS A SIMPLE EXAMPLE"
pat2 = "EXAMPLE"
print(f"txt = '{txt2}' (n={len(txt2)})")
print(f"pat = '{pat2}' (m={len(pat2)})\n")

sk2 = build_bm_skip(pat2)
print("이동량 표:")
for ch in "EXAMPL":
    k = pat2.rfind(ch)
    print(f"  '{ch}' → 마지막 출현 k={k}, 이동량 = {len(pat2)}-{k}-1 = {sk2[ord(ch)]}")
print(f"  그 외 글자 → {len(pat2)}")

print("\n라운드별 추적:")
m2 = len(pat2); pt = m2 - 1; rnd = 0; total_cmp = 0
while pt < len(txt2):
    rnd += 1; pp = m2 - 1; matched = 0
    while txt2[pt] == pat2[pp]:
        total_cmp += 1; matched += 1
        if pp == 0:
            print(f"  R{rnd}: 전부 일치! return {pt}")
            pt = -999; break
        pt -= 1; pp -= 1
    if pt == -999: break
    total_cmp += 1
    a = sk2[ord(txt2[pt])]; b = m2 - pp
    mv = a if a > b else b
    print(f"  R{rnd}: txt[{pt}]='{txt2[pt]}' 불일치 (뒤 {matched}글자 맞춤) → {mv}칸 이동")
    pt += mv

print(f"\n총 라운드 {rnd}회, 총 비교 {total_cmp}회 (n={len(txt2)})")
print(f"결과: {bm_match(txt2, pat2)}  → '{txt2[bm_match(txt2,pat2):]}'")
print(f"\n💡 n={len(txt2)}인데 비교는 {total_cmp}회뿐! 텍스트를 다 안 봤다 🔥")

### 6. 🔴 [설명] 🔥 "텍스트를 다 안 본다"의 의미

5번 결과를 잘 봐. 텍스트가 24글자인데 비교 횟수가 그보다 **적어**.

- 브루트 포스와 KMP는 최소 몇 번 비교해야 할까? 텍스트의 **모든 글자**를 최소 한 번씩은 봐야 하니 최소 **n번**이야.
- 그런데 보이어·무어는 **n보다 적게** 비교할 수 있어. 어떻게?
  💡 패턴 마지막 글자에서 불일치가 나면 `m`칸을 점프하는데, 그 사이 `m-1`개 글자는 **쳐다보지도 않아.**
- 이게 **O(n/m)** 의 직관이야:
  - 한 라운드에 **비교 1번** 하고 **m칸 점프**
  - 텍스트를 다 훑으려면 라운드가 `n / m` 번 필요
  - 총 비교 = `n / m` 번
- 🔥 **m이 커질수록 빨라지는** 유일한 알고리즘이야. 다른 건 m이 커지면 느려지거나 그대로인데.

**직접 계산해봐**: n = 100,000, m = 8, 패턴 글자가 텍스트에 하나도 없다면
- 브루트 포스 비교 횟수 ≈ ①____
- KMP 비교 횟수 ≈ ②____
- 보이어·무어 비교 횟수 ≈ ③____

*(계산한 뒤 15번에서 실측으로 확인!)*

---
# 📋 PART 3 — 건너뛰기 표 만들기 (7~10번)

> 교재 316~317p. KMP의 표보다 훨씬 단순해. **for문 두 개**면 끝나.

### 7. 🟢 [손] 교재 316p 규칙을 표로

교재가 정리한 이동량 규칙이야. 빈칸을 채워봐. (패턴 길이 = n, 여기선 헷갈리니 **m**으로 쓸게)

**패턴에 포함되지 않는 문자를 만난 경우**
> 패턴 이동량이 곧 ①____ 입니다.

**패턴에 포함되는 문자를 만난 경우**
> 마지막에 나오는 위치의 인덱스가 k이면 이동량은 ②________ 입니다.
> 같은 문자가 패턴 안에 중복해서 존재하지 않으면 패턴의 맨 끝 문자의 이동량은 ③____ 입니다.

**`pat = 'ABAC'` (m = 4)로 계산해봐:**

| 글자 | 마지막 출현 k | 이동량 |
|---|---|---|
| A | ④ | ⑤ |
| B | ⑥ | ⑦ |
| C | ⑧ | ⑨ |
| 그 외 (D~Z 등) | - | ⑩ |

**교재 [그림 7-10]과 대조해봐:**
```
A  B  C  D  E  F  G  ...
1  2  4  4  4  4  4  ...
```

🔥 **어? `C`가 4야?** 우리가 공식으로 계산하면 `4 - 3 - 1 = 0` 인데?
- 교재 본문은 **"맨 끝 문자의 이동량은 n"** 이라 했고, 그림도 4로 그렸어.
- 그런데 **코드**는 `skip[ord(pat[pt])] = len(pat) - pt - 1` 이라 `C = 0` 이 돼.
- **둘 중 뭐가 맞아?** 그리고 둘 다 정답을 낼까? → 10번에서 실측으로 확인해!

*(표를 채운 뒤 실행)*

In [ ]:
pat = PAT
m = len(pat)
print(f"pat = {pat}, m = {m}\n")
print("공식으로 계산:")
for ch in sorted(set(pat)):
    k = pat.rfind(ch)
    print(f"  '{ch}': 마지막 출현 k={k} → 이동량 = {m} - {k} - 1 = {m-k-1}")
print(f"  그 외: 이동량 = m = {m}")

print("\n코드가 만든 실제 표 (교재 그림 7-10 형식):")
sk = show_bm_skip(pat)

print(f"\n🔥 교재 그림은 C=4, 코드는 C={sk[ord('C')]}  → 10번에서 확인!")

### 8. 🟡 [손] 다른 패턴으로 표 만들기

각 패턴의 이동량 표를 손으로 만들어봐. (패턴에 나오는 글자만)

**(가) `ABC`** (m=3)

| 글자 | A | B | C | 그 외 |
|---|---|---|---|---|
| 이동량 | | | | |

**(나) `AAAA`** (m=4) 🔥

| 글자 | A | 그 외 |
|---|---|---|
| 이동량 | | |

- `A`의 이동량이 **0**이 나오지? 그럼 패턴이 전혀 안 움직이는데... 어떻게 될까?

**(다) `EXAMPLE`** (m=7)

| 글자 | E | X | A | M | P | L | 그 외 |
|---|---|---|---|---|---|---|---|
| 이동량 | | | | | | | |

**(라) `ABCDEFGH`** (m=8) — 중복 글자가 없는 패턴

| 글자 | A | B | C | D | E | F | G | H | 그 외 |
|---|---|---|---|---|---|---|---|---|---|
| 이동량 | | | | | | | | | |

- (라)처럼 **중복이 없고 m이 클수록** 이동량이 어떻게 돼? 검색이 빨라질까 느려질까?
- (나)처럼 **같은 글자만 반복**되면? 이게 BM의 최악 시나리오야 (18번).

*(채운 뒤 실행)*

In [ ]:
for p in ['ABC', 'AAAA', 'EXAMPLE', 'ABCDEFGH']:
    sk = build_bm_skip(p)
    m = len(p)
    uniq = sorted(set(p))
    print(f"pat = '{p}' (m={m})")
    print("  " + "  ".join(f"{c}" for c in uniq) + "  | 그외")
    print("  " + "  ".join(f"{sk[ord(c)]}" for c in uniq) + f"  |  {m}")
    print(f"  평균 이동량(패턴 글자): {sum(sk[ord(c)] for c in uniq)/len(uniq):.2f}\n")

print("→ 중복 없고 m 클수록 이동량이 크다 = 빠르다")
print("→ 같은 글자만 반복되면 이동량이 0에 가깝다 = 느리다")

### 9. 🟢 [설명] 왜 배열 크기가 256일까

```python
skip = [None] * 256
...
skip[ord(pat[pt])] = len(pat) - pt - 1
```

교재 317p: **"패턴 안에 존재할 수 있는 모든 문자의 이동량을 계산해야 하므로 이 건너뛰기 표에서 사용하는 원소는 256개입니다."**

보충수업 7-2:
> **"`ord()`는 단일한 문자를 전달받아 그 문자의 유니코드 코드 포인트를 정수로 반환합니다. 예를 들어 `ord('a')`는 정수 97을 반환합니다."**
> **"이 함수의 변환을 거꾸로 수행하는 내장 함수는 `chr()`입니다."**

- `ord('A')`, `ord('Z')`, `ord('a')`, `ord('0')` 은 각각 몇일까? 예측해봐.
- 256은 어디서 온 숫자야? (힌트: 1바이트로 표현 가능한 값의 개수)
- 🔥 **한글을 검색하면 어떻게 될까?** `ord('가')` 는 44032야. 그럼 `skip[44032]` 는?
- 이건 **25일차 도수 정렬**의 `f = [0] * (max + 1)` 과 똑같은 구조야. "값을 인덱스로" 쓰는 거지.
  - 도수 정렬은 값 범위가 크면 배열이 터졌어 (25일차 19번).
  - BM도 마찬가지 문제가 있어. 어떻게 해결할까? (딕셔너리를 쓰면?)

*(예측을 적은 뒤 실행)*

In [ ]:
print("[ord / chr 확인]")
for c in ['A', 'Z', 'a', 'z', '0', ' ']:
    print(f"  ord('{c}') = {ord(c):5d}   chr({ord(c)}) = '{chr(ord(c))}'")

print(f"\n  256 = 2^8 = 1바이트로 표현 가능한 값의 개수 (ASCII 확장)")
print(f"  교재 실습 7-3 주석: '문자열의 값은 0~255개'\n")

print("[한글을 넣으면?]")
print(f"  ord('가') = {ord('가')}")
try:
    bm_match("안녕하세요 반갑습니다", "반갑")
except IndexError as e:
    print(f"  IndexError: {e}")
    print("  → skip 배열 크기 256을 넘어간다! 🔥")

print("\n[딕셔너리로 고치면 해결]")
def bm_match_dict(txt, pat):
    m = len(pat)
    if m == 0: return 0
    skip = {}
    for i in range(m):
        skip[pat[i]] = m - i - 1
    pt = m - 1
    while pt < len(txt):
        pp = m - 1
        while txt[pt] == pat[pp]:
            if pp == 0: return pt
            pt -= 1; pp -= 1
        a = skip.get(txt[pt], m); b = m - pp
        pt += a if a > b else b
    return -1

print(f"  bm_match_dict('안녕하세요 반갑습니다', '반갑') = {bm_match_dict('안녕하세요 반갑습니다', '반갑')}")
print(f"  확인: '안녕하세요 반갑습니다'[6:8] = '{'안녕하세요 반갑습니다'[6:8]}'")
print("  → 25일차 19번에서 본 '값 범위가 크면 배열이 터진다' 문제의 해법과 같다")

### 10. 🔴 [실험] 🔥 교재 그림과 코드가 다르다

7번에서 발견한 것: 교재 [그림 7-10]은 `C = 4`, 코드는 `C = 0`.

- 어느 쪽이 **정답을 내는지** 실험으로 확인해봐. 랜덤 2만 개 케이스로 돌려볼 거야.
- **예측해봐**: 둘 다 맞을까? 하나만 맞을까?
- 만약 둘 다 맞다면, **왜** 그럴까? 🔥
  💡 힌트: 코드의 이동량 계산을 봐.
  ```python
  pt += skip[ord(txt[pt])] if skip[ord(txt[pt])] > len(pat) - pp else len(pat) - pp
  ```
  `skip` 값이 작아도 **`len(pat) - pp`** 라는 하한선이 있어. 이게 뭘 보장하지?
- 그리고 애초에 **패턴의 마지막 글자에서 불일치가 났을 때** `txt[pt]`가 그 마지막 글자일 수 있어? (일치했으면 안쪽으로 들어갔을 텐데?)

*(예측을 적은 뒤 실행)*

In [ ]:
def bm_variant(txt, pat, last_char_n=False):
    m = len(pat)
    skip = [m] * 256
    for i in range(m):
        skip[ord(pat[i])] = m - i - 1
    if last_char_n and pat.count(pat[-1]) == 1:
        skip[ord(pat[-1])] = m          # 교재 그림 7-10 방식
    pt = m - 1; guard = 0
    while pt < len(txt):
        guard += 1
        if guard > 100000: return 'LOOP'
        pp = m - 1
        while txt[pt] == pat[pp]:
            if pp == 0: return pt
            pt -= 1; pp -= 1
        a = skip[ord(txt[pt])]; b = m - pp
        pt += a if a > b else b
    return -1

random.seed(0)
fail_code = fail_book = 0
for _ in range(20000):
    t = ''.join(random.choice('ABC') for _ in range(random.randint(1, 18)))
    p = ''.join(random.choice('ABC') for _ in range(random.randint(1, 5)))
    e = naive(t, p)
    if bm_variant(t, p, False) != e: fail_code += 1
    if bm_variant(t, p, True) != e: fail_book += 1

print(f"코드 방식 (마지막 글자 = 0) 오답: {fail_code}/20000")
print(f"교재 그림 방식 (마지막 글자 = m) 오답: {fail_book}/20000")
print("\n→ 둘 다 정답! 왜?")
print("\n[이유 1] 마지막 글자에서 불일치할 때는 그 글자가 될 수 없다")
print("  pp = m-1 에서 txt[pt] == pat[m-1] 이면 안쪽 while로 들어간다.")
print("  즉 불일치 시점의 txt[pt]는 pat[m-1]과 '다른' 글자다.")
print("  → skip[pat[m-1]] 값은 pp = m-1 상황에서 읽히지 않는다\n")
print("[이유 2] max(skip, m-pp) 하한선이 안전을 보장한다")
p = PAT; m = len(p); sk = build_bm_skip(p)
print(f"  pat = {p}, skip['C'] = {sk[ord('C')]}")
print(f"  만약 pp=1 에서 txt[pt]='C' 로 불일치했다면:")
print(f"    a = skip['C'] = {sk[ord('C')]}, b = m - pp = {m} - 1 = {m-1}")
print(f"    → max = {max(sk[ord('C')], m-1)} 이 선택되어 최소 {m-1}칸은 전진 ✅")
print("  → skip 값이 0이어도 절대 제자리걸음하지 않는다")

---
# 💻 PART 4 — 코드 구현 (11~14번)

### 11. 🟢 [설명] 이동량 계산식 해부

```python
pt += skip[ord(txt[pt])] if skip[ord(txt[pt])] > len(pat) - pp \
    else len(pat) - pp
```

이 한 줄이 오늘 코드의 심장이야. 풀어 쓰면:

```python
a = skip[ord(txt[pt])]      # 나쁜 문자 규칙이 말하는 이동량
b = len(pat) - pp           # 최소 보장 이동량
pt += max(a, b)
```

**두 값의 의미를 각각 설명해봐:**
- `a = skip[txt[pt]]`: ①________________
- `b = len(pat) - pp`: ②________________

🔥 **`b`가 왜 필요한가** — 세 가지 이유를 생각해봐:
1. `a`가 **0**일 수 있어 (8번의 `AAAA`). 그럼 `pt`가 안 움직여서 ③________
2. `pp`가 안쪽으로 들어간 상태(`pp < m-1`)면 `pt`도 **되돌아가 있어.** `a`만 쓰면 `pt`가 원래 위치보다 **뒤로** 갈 수도 있어.
3. `b = len(pat) - pp` 는 "패턴을 최소 1칸 밀었을 때 `pt`가 가야 할 위치"야. 왜 그런지 계산해봐:
   - 현재 `pt`는 패턴의 `pp` 위치와 마주 봐.
   - 패턴을 1칸 밀면 패턴 끝(`m-1`)이 오는 자리는? → `pt + (m-1-pp) + 1 = pt + m - pp` ✅

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT; m = len(pat); sk = build_bm_skip(pat)
print(f"pat = {pat}, m = {m}, skip = {{A:{sk[ord('A')]}, B:{sk[ord('B')]}, C:{sk[ord('C')]}, 그외:{m}}}\n")

print("상황별 이동량 계산:")
cases = [
    ('X', 3, "패턴에 없는 글자, 첫 비교에서 불일치"),
    ('A', 3, "패턴에 있는 글자(마지막 출현 k=2), 첫 비교"),
    ('Z', 2, "패턴에 없는 글자, 1글자 맞춘 뒤 불일치"),
    ('B', 1, "패턴에 있는 글자, 2글자 맞춘 뒤 불일치"),
]
for ch, pp, desc in cases:
    a = sk[ord(ch)] if ord(ch) < 256 else m
    b = m - pp
    print(f"  txt[pt]='{ch}', pp={pp} ({desc})")
    print(f"    a = skip['{ch}'] = {a}, b = m - pp = {m}-{pp} = {b} → 이동량 {max(a,b)}")

print("\n[b가 없으면 무슨 일이?]")
print("  pat='AAAA' 일 때 skip['A'] = 0")
def bm_no_floor(txt, pat, limit=1000):
    m = len(pat); skip = [m]*256
    for i in range(m): skip[ord(pat[i])] = m-i-1
    pt = m-1; g = 0
    while pt < len(txt):
        g += 1
        if g > limit: return f"❗ 무한 루프 (pt={pt})"
        pp = m-1
        while txt[pt] == pat[pp]:
            if pp == 0: return pt
            pt -= 1; pp -= 1
        pt += skip[ord(txt[pt])]        # 🐛 하한선 없음
    return -1
print(f"  bm_no_floor('AAABAAA', 'AAAA') = {bm_no_floor('AAABAAA', 'AAAA')}")

### 12. 🔴 [디버깅] 🔥 `pt` 초기화가 없다

```python
def bm_match(txt, pat):
    skip = [None] * 256
    for pt in range(256):
        skip[pt] = len(pat)
    for pt in range(len(pat)):              # ← 이 for문이 끝나면 pt는?
        skip[ord(pat[pt])] = len(pat) - pt - 1

    while pt < len(txt):                    # ← pt를 초기화한 적이 없다!
```

- `pt = ...` 형태의 **초기화 문장이 어디에도 없어.** 그런데 코드는 잘 동작해. 왜?
- 파이썬에서 `for pt in range(len(pat)):` 가 끝나면 `pt` 의 값은? ①____
- 보이어·무어는 **패턴의 끝 글자**를 텍스트의 어느 위치에 맞추고 시작해야 하지? ②____
- 두 값이 **우연히 같아서** 동작하는 거야. 🔥
- ⚠️ 이게 왜 위험할까? 세 가지를 생각해봐:
  - 표 만들기 코드를 함수로 분리하면?
  - `len(pat) == 0` 이면?
  - 두 `for` 문의 순서를 바꾸면?
- 💡 22일차 14번 (`while j > 0`이 우연히 안전했던 것)과 **같은 계열**이야. **"동작하지만 이유가 코드 밖에 있는"** 코드.

*(예측을 적은 뒤 실행)*

In [ ]:
print("[1] for문이 끝난 뒤 변수는 살아 있다")
for pt in range(4):
    pass
print(f"  for pt in range(4) 종료 후 → pt = {pt}  (= 4-1)")
print(f"  BM은 pt = len(pat)-1 에서 시작해야 함 → 우연히 일치 ✅\n")

print("[2] 표 만들기를 함수로 분리하면?")
def bm_broken(txt, pat):
    skip = build_bm_skip(pat)      # 표를 함수로 분리
    while pt < len(txt):           # 🐛 pt가 정의되지 않음
        pp = len(pat) - 1
        while txt[pt] == pat[pp]:
            if pp == 0: return pt
            pt -= 1; pp -= 1
        a = skip[ord(txt[pt])]; b = len(pat) - pp
        pt += a if a > b else b
    return -1
try:
    bm_broken('ABCXDEZCABACABAC', 'ABAC')
except (NameError, UnboundLocalError) as e:
    print(f"  {type(e).__name__}: {e}")
    print("  → 리팩터링하는 순간 터진다 🔥\n")

print("[3] 빈 패턴이면?")
def bm_empty_test(pat):
    skip = [None]*256
    for pt in range(256): skip[pt] = len(pat)
    for pt in range(len(pat)): pass
    return pt
print(f"  len(pat)=0 → 두 번째 for가 한 번도 안 돌아 pt는 첫 for의 값 {bm_empty_test('')} 로 남는다")
print(f"  → 의도와 전혀 다른 값! (안전하려면 명시적 초기화 필요)\n")

print("[올바르게 고친 버전]")
def bm_fixed(txt: str, pat: str) -> int:
    m = len(pat)
    if m == 0: return 0
    skip = build_bm_skip(pat)
    pt = m - 1                      # ✅ 명시적 초기화
    while pt < len(txt):
        pp = m - 1
        while txt[pt] == pat[pp]:
            if pp == 0: return pt
            pt -= 1; pp -= 1
        a = skip[ord(txt[pt])]; b = m - pp
        pt += a if a > b else b
    return -1
print(f"  bm_fixed('{TXT}', '{PAT}') = {bm_fixed(TXT, PAT)}")

### 13. 🟡 [빈칸] `bm_match` 직접 구현

7~12번에서 뜯어본 걸 조립해. (12번에서 배운 대로 **`pt`를 명시적으로 초기화**할 것)

**기대 출력**
```
8
2
-1
랜덤 1000회 검증: 실패 0회
```

In [ ]:
def my_bm_match(txt: str, pat: str) -> int:
    """보이어·무어법으로 문자열 검색"""
    m = len(pat)
    if m == 0: return 0

    # --- 건너뛰기 표 만들기 ---
    skip = [___] * 256                    # ① 패턴에 없는 글자의 기본 이동량
    for i in range(m):
        skip[ord(pat[i])] = ___           # ② 마지막 출현 기준 이동량 (2번 공식)

    # --- 검색하기 ---
    pt = ___                              # ③ 어디서 시작? (12번)
    while pt < len(txt):
        pp = ___                          # ④ 패턴의 어느 위치부터 비교?
        while txt[pt] == pat[pp]:
            if pp == ___:                 # ⑤ 전부 맞췄다는 조건
                return ___                # ⑥ 이때 pt가 곧 답
            pt -= 1
            pp -= 1
        a = skip[ord(txt[pt])]            # 나쁜 문자 규칙
        b = ___                           # ⑦ 최소 보장 이동량 (11번)
        pt += ___                         # ⑧ 둘 중 어느 쪽?
    return -1


print(my_bm_match('ABCXDEZCABACABAC', 'ABAC'))
print(my_bm_match('ABABCDEFGHA', 'ABC'))
print(my_bm_match('ABCDEF', 'XYZ'))

fail = 0
for _ in range(1000):
    t = ''.join(random.choice('ABC') for _ in range(random.randint(0, 20)))
    p = ''.join(random.choice('ABC') for _ in range(random.randint(1, 5)))
    if my_bm_match(t, p) != naive(t, p): fail += 1
print(f"랜덤 1000회 검증: 실패 {fail}회")

### 14. 🟡 [설명] `return pt` — 왜 `pt - pp` 가 아닐까

26일차와 27일차는 이랬어:
```python
return pt - pp if pp == len(pat) else -1
```

그런데 오늘은:
```python
if pp == 0:
    return pt
```

- 왜 `pt` 하나만으로 충분해? 🔥
  💡 힌트: `pp == 0` 이라는 건 패턴의 **어느 글자**까지 비교했다는 뜻이지?
- BM은 뒤에서부터 비교하니까, `pp == 0` 인 순간 `pt` 는 **매칭 구간의 어디**를 가리켜?
- 26일차 불변식 `pt - pp` 를 여기에 적용하면? `pt - 0 = pt` — 사실 **같은 공식**이야!
- 그럼 **왜 `-1` 판정도 다를까**? BM은 `while pt < len(txt)` 를 빠져나오면 무조건 실패지. 26~27일차처럼 `pp` 로 판정할 필요가 없는 이유는?

*(답을 적은 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
m = len(pat)
r = bm_match(txt, pat)
print(f"결과 = {r}")
print(f"  pp == 0 인 순간 pt = {r}")
print(f"  BM은 뒤→앞으로 비교하므로 pp=0 이면 pt는 매칭의 '시작' 위치 ✅")
print(f"  txt[{r}:{r+m}] = '{txt[r:r+m]}'\n")

print("[세 알고리즘의 반환값 비교]")
print(f"  BF  : return pt - pp  (pp = m 일 때, pt는 매칭의 '끝 다음')")
print(f"  KMP : return pt - pp  (동일)")
print(f"  BM  : return pt       (pp = 0 일 때, pt가 곧 매칭의 '시작')")
print(f"  → 셋 다 사실 'pt - pp' 공식. BM은 pp가 0이라 생략된 것뿐!\n")

print("[세 알고리즘 결과 일치 검증]")
random.seed(5); fail = 0
for _ in range(3000):
    t = ''.join(random.choice('ABCD') for _ in range(random.randint(0, 25)))
    p = ''.join(random.choice('ABCD') for _ in range(random.randint(1, 6)))
    a, b, c, d = bf_match(t,p), kmp_match(t,p), bm_match(t,p), naive(t,p)
    if not (a == b == c == d): fail += 1
print(f"  랜덤 3000회: 불일치 {fail}회 ✅")

---
---

# ⏱️ PART 5 — 시간 복잡도 완전 정복 (15~19번)

> **오늘의 두 번째 목표.** 교재 319p가 이렇게 정리해:
>
> - **브루트 포스법**: O(mn) — 다만 일부러 꾸며 낸 패턴이 아니라면 O(n)
> - **KMP법**: 최악의 경우에도 **O(n)**
> - **보이어·무어법**: 최악이라도 O(n), 평균 **O(n / m)**
>
> **"왜?"** 를 이제 하나씩 손으로 세고 눈으로 확인한다.

## 먼저 용어 정리
- **n** = 텍스트 길이, **m** = 패턴 길이
- **"비교 횟수"** = `txt[?] == pat[?]` 를 실행한 횟수
- 이 파트에서는 **못 찾는 경우**(끝까지 훑는 경우)를 기준으로 세. 그래야 상한이 보이거든.

### 15. 🟢 [계산] 손으로 세어보기 — 아주 작은 예제

먼저 **직접 세는** 것부터. 계산기 없이 종이로.

```
txt = A A A A A A A A A B     (n = 10)
pat = A A B                   (m = 3)
```

**(가) 브루트 포스** (26일차)
- 시작 위치 0에서: `A==A` ✅, `A==A` ✅, `A==B` ❌ → **3번** 비교
- 시작 위치 1에서: **____번**
- ...
- 시작 위치는 총 **____개** (0부터 `n-m`까지)
- 각 위치에서 **최대 m번** 비교
- **총 비교 ≈ ①____ × ②____ = ③____**
- 👉 이게 **O(m·n)** 의 정체: **"시작 위치마다 최대 m번씩"**

**(나) KMP** (27일차)
- `pt`는 **줄어들지 않아.** 최대 몇 번 증가할 수 있어? ④____
- `pp`는 일치할 때 +1, `skip` 참조할 때 감소해. 총 증가량이 `n`을 못 넘으니 **총 감소량도** ⑤____ 이하
- **총 비교 ≤ ⑥____**
- 👉 이게 **O(n)** 의 정체: **"두 커서의 총 이동량이 n에 비례"**

**(다) 보이어·무어** (오늘)
- 라운드 1: `txt[2]='A'` vs `pat[2]='B'` → 불일치. `skip['A']` = ⑦____ → `pt` 이동
- 각 라운드에서 **최소 1번** 비교하고 **평균 ⑧____ 칸** 점프
- 라운드 수 ≈ ⑨____
- **총 비교 ≈ ⑩____**
- 👉 이게 **O(n / m)** 의 정체: **"m칸씩 건너뛰니 라운드가 n/m번"**

*(계산한 뒤 실행해서 대조)*

In [ ]:
t, p = "AAAAAAAAAB", "AAB"
n, m = len(t), len(p)
print(f"txt = {t} (n={n}), pat = {p} (m={m})\n")

print("[브루트 포스] 시작 위치별 비교 횟수")
total = 0
for i in range(n - m + 1):
    c = 0
    for j in range(m):
        c += 1
        if t[i+j] != p[j]: break
    total += c
    print(f"  i={i}: {c}번")
print(f"  합계 {total}번   (n-m+1={n-m+1} 개 위치 × 최대 m={m}번 = 최대 {(n-m+1)*m}번)")
print(f"  실제 함수 측정: {bf_count(t,p)}번\n")

print(f"[KMP] 측정: {kmp_count(t,p)}번   (n={n} 근처)")
print(f"[BM ] 측정: {bm_count(t,p)}번   (n/m={n/m:.1f} 근처)")

### 16. 🟡 [실험] 🔥 O(n/m)을 눈으로 — m을 키워보자

**여기가 오늘의 핵심 실험이야.**

n = 100,000으로 **고정**하고, **m만 키워가며** 비교 횟수를 재볼 거야.
패턴은 텍스트에 **절대 없는 글자**(`Z`)로만 만들어서 끝까지 훑게 할 거고.

**예측해봐:**

| m | 브루트 포스 | KMP | 보이어·무어 |
|---|---|---|---|
| 2 | ① | ② | ③ |
| 8 | ④ | ⑤ | ⑥ |
| 64 | ⑦ | ⑧ | ⑨ |

- m이 커지면 각각 **늘어날까, 줄어들까, 그대로일까?**
- 🔥 **셋 중 하나만 m이 커질수록 빨라져.** 어느 거야?

*(예측을 적은 뒤 실행 — 조금 걸려)*

In [ ]:
random.seed(1)
N = 100000
# 텍스트는 A~Y만 사용, 패턴은 Z로 → 절대 매칭 안 됨
txt_big = ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXY') for _ in range(N))

print(f"[n = {N:,} 고정, m만 키우기 — 패턴은 텍스트에 없음]\n")
print("   m  |     BF      |     KMP     |     BM      |    n/m    | BM ÷ (n/m)")
print("-" * 76)
for m in (2, 4, 8, 16, 32, 64):
    p = 'Z' * m
    cb = bf_count(txt_big, p)
    ck = kmp_count(txt_big, p)
    cm = bm_count(txt_big, p)
    print(f"  {m:3d} | {cb:11,d} | {ck:11,d} | {cm:11,d} | {N//m:9,d} |   {cm/(N/m):.2f}")

print("\n🔥 관찰")
print("  · BF, KMP: m이 커져도 비교 횟수가 n으로 '고정' → O(n)")
print("  · BM: m이 2배가 될 때마다 비교가 '절반'으로 → O(n/m)")
print("  · 마지막 열이 정확히 1.00 → BM 비교 횟수 = n/m 그 자체!")

### 17. 🟡 [실험] n을 키워보자

이번엔 **m = 8로 고정**하고 **n만** 키워볼게.

**예측해봐:** n이 2배가 되면 세 알고리즘의 비교 횟수는 각각 몇 배가 될까?

| | n 2배 → 비교 횟수 |
|---|---|
| 브루트 포스 | ① |
| KMP | ② |
| 보이어·무어 | ③ |

- 셋 다 **n에 비례**해서 늘어날 거야. 차이는 **계수(비례상수)** 지.
- 그럼 O(n)과 O(n/m)의 차이는 결국 뭐야? ④________________

*(예측을 적은 뒤 실행)*

In [ ]:
print(f"[m = 8 고정, n만 키우기]\n")
print("      n    |     BF      |     KMP     |     BM      | BM ÷ n")
print("-" * 66)
prev = None
for n in (10000, 25000, 50000, 100000, 200000):
    t = ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXY') for _ in range(n))
    p = 'Z' * 8
    cb, ck, cm = bf_count(t, p), kmp_count(t, p), bm_count(t, p)
    print(f" {n:8,d}  | {cb:11,d} | {ck:11,d} | {cm:11,d} |  {cm/n:.4f}")

print("\n🔥 관찰")
print("  · 셋 다 n에 정비례한다 (n이 2배면 비교도 2배)")
print("  · BF/KMP의 계수는 1.0, BM의 계수는 1/m = 1/8 = 0.125")
print("  · 즉 O(n)과 O(n/m)의 차이는 '기울기'다")
print("\n💡 Big-O 표기에서 O(n/m) 은 m을 상수로 보면 O(n) 이지만,")
print("   m이 함께 커지는 상황에서는 결정적인 차이가 된다.")

### 18. 🔴 [실험] 🔥 교재가 말하지 않은 것 — BM의 진짜 최악

교재 319p는 보이어·무어를 **"최악의 경우라도 O(n)"** 이라고 해.
그런데 3번에서 본 각주를 기억해?

> **"여기에서는 배열 하나를 사용하여 보이어·무어법을 간략하게 나타냈습니다. 원래의 보이어·무어법은 배열 2개를 사용해서 검사합니다."**

우리가 배운 건 **나쁜 문자 규칙만 쓰는 간이 버전**이야. **최악 보장은 두 번째 표(착한 접미사 규칙)가 있어야 성립해.**

**최악 시나리오를 만들어보자:**
```
txt = A A A A A ... A       (전부 A)
pat = B A A A A A ... A     (앞만 B)
```
- 매 라운드마다 **뒤에서 m-1글자가 전부 일치**하다가 **맨 앞 `B`에서 불일치**해.
- 그때 `txt[pt] = 'A'`, `skip['A']` = ①____ (패턴에서 A의 마지막 출현은 m-1)
- 그리고 `b = m - pp = m - 0 = m`... 이 아니라 잠깐, `pp = 0` 이면? 계산해봐.
- 결국 **한 라운드에 m번 비교하고 겨우 몇 칸만** 전진해.

**예측**: 이 경우 BM의 비교 횟수는 KMP보다 많을까 적을까?

*(예측을 적은 뒤 실행)*

In [ ]:
print("[최악 케이스] txt = 'AAA...A', pat = 'BAA...A'\n")
print("      n | m  |     BF      |     KMP     |     BM      | BM ÷ n")
print("-" * 70)
for n, m in ((10000, 10), (20000, 10), (10000, 50), (10000, 100)):
    t = 'A' * n
    p = 'B' + 'A' * (m - 1)
    cb, ck, cm = bf_count(t, p), kmp_count(t, p), bm_count(t, p)
    print(f" {n:6,d} | {m:3d} | {cb:11,d} | {ck:11,d} | {cm:11,d} | {cm/n:7.1f}")

print("\n🔥 BM이 셋 중 가장 느리다!")
print("  · BM 비교 횟수 ÷ n 이 m에 비례해서 커진다 → O(n·m)")
print("  · 교재가 말한 '최악 O(n)'은 배열 2개짜리 정식 BM 이야기")
print("  · 실습 7-3(배열 1개)은 최악 O(n·m) 이다\n")

print("[왜 이렇게 되나]")
m = 10
p = 'B' + 'A'*(m-1)
sk = build_bm_skip(p)
print(f"  pat = '{p}'")
print(f"  skip['A'] = {sk[ord('A')]}  ('A'의 마지막 출현이 인덱스 {p.rfind('A')} 이므로 {m}-{p.rfind('A')}-1)")
print(f"  매 라운드: 뒤에서 {m-1}글자 일치 → pp=0 에서 'B' vs 'A' 불일치")
print(f"    a = skip['A'] = {sk[ord('A')]}, b = m - pp = {m} - 0 = {m}")
print(f"    → 이동량 {max(sk[ord('A')], m)} 이지만, 그 대가로 {m}번을 비교했다")
print(f"  → 비교 {m}번 ÷ 이동 {m}칸 = 글자당 1번... 이 아니라,")
print(f"     pt가 안쪽으로 {m-1}칸 되돌아갔다가 다시 나오므로 실제로는 더 손해")

### 19. 🟢 [정리] 문자열 검색 3종 최종 총정리

07장을 다 배웠어. 표를 완성해봐.

| | 비교 방향 | 불일치 시 동작 | 전처리 | 평균 | 최악 | 실무 사용 |
|---|---|---|---|---|---|---|
| 브루트 포스 (26일) | 앞→뒤 | `pt` 되돌리고 `pp=0` | 없음 | ① | ② | 짧은 입력에 OK |
| KMP (27일) | 앞→뒤 | `pp = skip[pp]` | ③ | ④ | ⑤ | ⑥ 거의 안 씀 |
| 보이어·무어 (오늘) | ⑦ | ⑧ | ⑨ | ⑩ | ⑪ | ⑫ |

**교재 319p의 결론:**
> **"일반적으로 파이썬에서 문자열 검색을 하려면 표준 라이브러리를 사용하는 것을 추천합니다. 만약 표준 라이브러리를 사용하지 않는다면 보이어·무어법(또는 개선한 방법)이나 상황에 따라서 브루트 포스법을 사용하는 경우가 많습니다."**

**최종 질문 4개**
1. **왜 O(n/m)이 O(n)보다 빠른가?** m이 클수록 어떻게 되는지 15~16번 실측으로 답해봐.
2. **BM이 "텍스트를 다 안 본다"는 게 어떻게 가능해?** 브루트 포스/KMP는 왜 불가능하지?
3. 교재 319p는 KMP에 대해 **"검색 과정에서 주목하는 곳을 앞으로 되돌릴 필요가 전혀 없으므로 파일을 차례로 읽어 들이면서 검색할 때 사용하면 좋습니다"** 라고 해. BM은 이게 가능할까? 왜?
4. 그럼 결국 **언제 뭘 써야** 할까? 세 가지 상황을 적어봐.

*(답을 적은 뒤 실행)*

In [ ]:
print("[최종 실측 종합]  같은 조건에서 세 알고리즘\n")
random.seed(3)
scenarios = [
    ("평범한 영문 텍스트", ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXY') for _ in range(100000)), 'ZZZZZZZZ'),
    ("알파벳 2종 (DNA 같은)", ''.join(random.choice('AB') for _ in range(100000)), 'ABABABAB'),
    ("최악 (반복 문자)", 'A'*100000, 'B' + 'A'*9),
]
print(f"{'상황':22s} | {'BF':>10s} | {'KMP':>10s} | {'BM':>10s} | 승자")
print("-" * 74)
for name, t, p in scenarios:
    cb, ck, cm = bf_count(t, p), kmp_count(t, p), bm_count(t, p)
    best = min([('BF', cb), ('KMP', ck), ('BM', cm)], key=lambda x: x[1])[0]
    print(f"{name:22s} | {cb:10,d} | {ck:10,d} | {cm:10,d} | {best}")

print("\n[실제 시간도 재보자 — str.find 포함]")
t = ''.join(random.choice('ABCDEFGHIJKLMNOPQRSTUVWXY') for _ in range(200000))
p = 'ZZZZZZZZ'
for name, fn in (("bf_match", bf_match), ("kmp_match", kmp_match),
                 ("bm_match", bm_match), ("str.find", lambda a, b: a.find(b))):
    t0 = time.perf_counter(); r = fn(t, p); el = time.perf_counter() - t0
    print(f"  {name:10s}: {el:.4f}s → {r}")

print("\n💡 str.find(Two-way)가 여전히 가장 빠르다 — C 구현 + 하이브리드 알고리즘")

---
---

# ✅ 정답 & 해설

> ⚠️ **표를 손으로 채운 뒤에 내려와.** 특히 4·15번은 직접 세봐야 남아.

---

## 🔁 Remind

### R-1
| 알고리즘 | 불일치 시 |
|---|---|
| 브루트 포스 | ① **`pt`를 되돌리고 시작 위치를 1칸 민다** |
| KMP | ② **`pt`는 그대로, `pp`만 표대로 줄인다** |
| 보이어·무어 | ③ **표를 보고 `pt`를 여러 칸 앞으로 점프** |

- 26일차 불변식 `pt - pp` = **현재 대조의 시작 위치**. 오늘도 쓰여 — 14번에서 `return pt` 가 사실 `pt - 0` 이라는 걸 볼 거야.

### R-2
| | KMP의 skip | BM의 skip |
|---|---|---|
| 인덱스 | 패턴에서 맞춘 글자 수 | ② **글자의 유니코드 값 (`ord`)** |
| 값 | 되돌아갈 `pp` | ④ **패턴을 오른쪽으로 밀 칸 수** |
| 크기 | `len(pat)+1` | ⑥ **256 (문자 종류 수)** |

**완전히 다른 물건**이야. KMP는 "패턴 위치 → 패턴 위치", BM은 "글자 → 이동량".

---

## 🎯 PART 1 해설

### 1. 패턴에 없는 글자
- 패턴 마지막 글자는 `txt[3]` 과 마주 봐 (m=4이므로 인덱스 3).
- ① **X**, ② **없다**
- **1~3칸 밀어도 소용없는 이유**: 패턴을 1칸 밀면 `txt[3]='X'` 가 `pat[2]='A'` 와, 2칸이면 `pat[1]='B'` 와, 3칸이면 `pat[0]='A'` 와 마주 봐. **패턴에 `X`가 아예 없으니 어느 자리와 맞춰도 불일치**야.
- 그래서 `X`를 **완전히 지나치는** 4칸 점프가 안전하면서 최대한의 이동.

> 🔑 **"이 글자가 패턴에 없다"는 정보 하나로 m-1개 위치를 한꺼번에 배제**하는 게 BM의 힘이야.

---

### 2. 패턴에 있는 글자
- **뒤쪽 `A`(인덱스 2)** 에 맞춰야 안전해.
- **앞쪽 `A`(인덱스 0)에 맞추면 안 되는 이유** 🔥: 3칸을 밀면 인덱스 2의 `A`가 텍스트에서 **이미 지나간 자리**로 가버려. 그 사이 위치에 정답이 있어도 **확인 없이 건너뛰게** 돼. 26일차 10번의 "근거 없이 여러 칸 밀기" 버그와 같은 실수지.
- **일반화**: 글자 `c`의 패턴 내 **마지막 출현 인덱스가 k**면 이동량 = **`m - k - 1`**
- 항상 "가장 뒤쪽 출현" 기준 = **보수적(가장 적게 미는) 선택**. 안전이 먼저야.

---

### 3. 이름 정리
- BM = **Boyer-Moore**. 우리가 배운 건 **나쁜 문자 규칙(bad character rule)** 만 쓰는 간이 버전.
- 정식 BM은 **착한 접미사 규칙(good suffix rule)** 이라는 두 번째 표를 더해. 이게 18번에서 결정적으로 중요해져.

---

## ✍️ PART 2 해설

### 4. 교재 그림 7-5~7-9 재현

| 라운드 | pt | pp | txt[pt] | pat[pp] | 결과 | 이동량 | 다음 pt |
|---|---|---|---|---|---|---|---|
| 1 | 3 | 3 | X | C | 불일치 | ① **4** | ② **7** |
| 2 | ③ **7** | 3 | ④ **C** | C | ⑤ **일치** → 안쪽으로 | | |
| | 6 | 2 | Z | A | 불일치 | max(4, 4-2=2)=**4** | **10** |
| 3 | 10 | 3 | A | C | 불일치 | max(skip[A]=1, 1)=**1** | **11** |
| 4 | 11 | 3 | C | C | 일치 → 안쪽 4글자 전부 일치 | | **return 8** |

- 라운드 2에서 `pt`가 7 → 6으로 **되돌아갔다가** 이동량 4를 더해 10이 됐지. BM의 `pt`는 **라운드 안에서는 뒤로, 라운드 사이에서는 앞으로** 움직여.
- 최종 `pt = 8`, `return 8`. `txt[8:12] = 'ABAC'` ✅

---

### 5. `EXAMPLE` 예제

| 글자 | E | X | A | M | P | L | 그 외 |
|---|---|---|---|---|---|---|---|
| 마지막 출현 k | ① **6** | ② **1** | ③ **2** | ④ **3** | ⑤ **4** | ⑥ **5** | - |
| 이동량 | ⑦ **0** | ⑧ **5** | ⑨ **4** | ⑩ **3** | ⑪ **2** | ⑫ **1** | ⑬ **7** |

🔥 **`E`의 이동량이 0**이야. `EXAMPLE`에서 `E`는 인덱스 0과 6에 나오는데 마지막 출현이 6(맨 끝)이라 `7-6-1 = 0`.
- 이동량 0이면 제자리걸음 → **무한 루프?** 아니야. 11번의 `b = m - pp` 하한선이 막아줘.

**실측**: n=24인데 **총 비교 약 15회**, 라운드 5회 정도로 끝나. 텍스트를 다 안 봤지!

---

### 6. 🔥 "텍스트를 다 안 본다"

- BF와 KMP는 **모든 글자를 최소 한 번씩** 봐야 해 → 최소 n번 비교.
- BM은 마지막 글자에서 불일치하면 m칸 점프하는데, 그 사이 **m-1개 글자는 쳐다보지도 않아.**
- **O(n/m) 직관**:
  - 한 라운드 = **비교 1번 + m칸 점프**
  - 텍스트를 훑는 데 필요한 라운드 = **n / m** 번
  - 총 비교 = **n / m** 번

**계산 (n=100,000, m=8)**
- ① BF ≈ **100,000**
- ② KMP ≈ **100,000**
- ③ BM ≈ **12,500** (= 100,000 / 8)

> 🔑 **BM은 m이 커질수록 빨라지는 유일한 알고리즘.** 다른 건 m이 커지면 그대로거나 느려져.

---

## 📋 PART 3 해설

### 7. 이동량 규칙

- ① **m** (패턴 길이)
- ② **m - k - 1**
- ③ **m**

`pat = 'ABAC'` (m=4):

| 글자 | 마지막 출현 k | 이동량 |
|---|---|---|
| A | ④ **2** | ⑤ **1** |
| B | ⑥ **1** | ⑦ **2** |
| C | ⑧ **3** | ⑨ **0** |
| 그 외 | - | ⑩ **4** |

🔥 **교재 그림은 C=4, 코드는 C=0.** 교재 본문의 "맨 끝 문자의 이동량은 n"과 코드의 `m-k-1` 공식이 어긋나. → 10번에서 **둘 다 정답**임을 확인해.

---

### 8. 다른 패턴들

| 패턴 | 이동량 |
|---|---|
| `ABC` (m=3) | A:2, B:1, C:0, 그외:3 |
| `AAAA` (m=4) | A:**0**, 그외:4 |
| `EXAMPLE` (m=7) | E:0, X:5, A:4, M:3, P:2, L:1, 그외:7 |
| `ABCDEFGH` (m=8) | A:7, B:6, C:5, D:4, E:3, F:2, G:1, H:0, 그외:8 |

- **중복 없고 m이 클수록** 이동량이 커 → **빠르다.** `ABCDEFGH`는 평균 이동량이 3.5나 돼.
- **같은 글자만 반복**(`AAAA`)되면 이동량이 0 → **느리다.** 18번의 최악 시나리오가 이 계열이야.

---

### 9. 왜 256인가

```
ord('A') = 65,  ord('Z') = 90,  ord('a') = 97,  ord('0') = 48,  ord(' ') = 32
```
- **256 = 2⁸** — 1바이트로 표현 가능한 값의 개수. 교재 실습 7-3 주석의 "문자열의 값은 0~255개"가 이 뜻이야.
- 🔥 **한글은 터진다**: `ord('가') = 44032` 라서 `skip[44032]` 는 `IndexError`. 실행 결과에서 확인했지.
- **25일차 도수 정렬과 같은 구조**야: "값을 인덱스로" 쓰는 방식. 25일차 19번에서 값 범위가 크면 배열이 터졌던 그 문제 그대로.
- **해결**: `dict` 로 바꾸면 돼. 실행 셀의 `bm_match_dict` 가 한글도 잘 처리하지. 실무 구현은 대부분 이렇게 해.

---

### 10. 🔥 교재 그림과 코드가 달라도 둘 다 맞는 이유

**실측: 코드 방식 오답 0/20000, 교재 그림 방식 오답 0/20000** — 둘 다 정답!

**이유 1**: **패턴의 마지막 글자에서 불일치할 때는 `txt[pt]`가 그 글자일 수 없어.**
`pp = m-1` 에서 `txt[pt] == pat[m-1]` 이면 안쪽 `while`로 들어가버리니까. 즉 `skip[pat[m-1]]` 값은 **`pp = m-1` 상황에서 절대 읽히지 않아.**

**이유 2**: `pp < m-1` 인 상황에서 읽힐 수는 있는데, 그때는 **`b = m - pp` 하한선**이 작동해.
```
pat='ABAC', pp=1 에서 txt[pt]='C' 불일치
  a = skip['C'] = 0,  b = m - pp = 4 - 1 = 3
  → max(0, 3) = 3칸 전진 ✅
```
`skip` 값이 0이어도 **절대 제자리걸음하지 않아.**

> 🔑 교재 본문/그림은 **개념 설명용**(맨 끝 글자는 밀 필요 없다 → n칸)이고, 코드는 **공식 일관성**(`m-k-1`)을 택한 거야. 하한선이 있어서 결과가 같아지지.

---

## 💻 PART 4 해설

### 11. 이동량 계산식

```python
a = skip[ord(txt[pt])]     # ① 나쁜 문자 규칙: 이 글자를 패턴의 마지막 출현에 맞추려면 몇 칸?
b = len(pat) - pp          # ② 최소 보장: 패턴을 1칸만 밀었을 때 pt가 가야 할 거리
pt += max(a, b)
```

**`b`가 필요한 세 가지 이유**
1. `a`가 **0**일 수 있어 (`AAAA`의 `A`, `EXAMPLE`의 `E`). 그럼 ③ **무한 루프**가 돼. 실행 셀의 `bm_no_floor` 가 실제로 무한 루프에 걸리지.
2. `pp < m-1` 이면 `pt` 가 **안쪽으로 되돌아간 상태**야. `a`만 쓰면 원래 라운드 시작 위치보다 **뒤로 갈 수도** 있어.
3. **`b = m - pp` 유도**:
   - 현재 `pt` 는 `pat[pp]` 와 마주 봄
   - 패턴을 1칸 밀면 패턴 끝(`m-1`)이 올 자리 = `pt + (m-1-pp) + 1` = **`pt + m - pp`** ✅

> 🔑 `max(a, b)` 는 **"최대한 많이 밀되, 최소 1칸은 반드시 민다"** 는 뜻이야.

---

### 12. 🔥 `pt` 초기화가 없다

```python
for pt in range(len(pat)):        # ← 이 루프가 끝나면
    skip[ord(pat[pt])] = ...
while pt < len(txt):              # ← pt = len(pat) - 1 인 채로 시작
```

- ① `pt = len(pat) - 1` (파이썬 `for` 변수는 루프 후에도 살아 있어)
- ② BM은 **패턴의 끝 글자를 `txt[m-1]`에 맞추고** 시작해야 함
- **두 값이 우연히 같아서** 동작하는 거야 🔥

**왜 위험한가 — 실측**
1. **표 만들기를 함수로 분리하면** → `UnboundLocalError`. 리팩터링하는 순간 터져.
2. **`len(pat) == 0`** 이면 두 번째 `for`가 안 돌아서 `pt`는 첫 `for`의 값 **255**로 남아. 완전히 엉뚱한 값.
3. 두 `for`문 순서를 바꾸면 역시 깨져.

**22일차 14번(`while j > 0`)과 같은 계열** — "동작하지만 이유가 코드 밖에 있는" 코드야. `pt = m - 1` 한 줄만 명시하면 전부 해결돼.

---

### 13. `my_bm_match`

```python
skip = [m] * 256                      # ①
skip[ord(pat[i])] = m - i - 1         # ②
pt = m - 1                            # ③  ← 명시적 초기화! (12번)
pp = m - 1                            # ④  ← 패턴 끝부터
if pp == 0:                           # ⑤
    return pt                         # ⑥
b = m - pp                            # ⑦
pt += a if a > b else b               # ⑧  (= max(a, b))
```
출력: `8`, `2`, `-1`, 랜덤 1000회 실패 0회

---

### 14. `return pt` — 왜 `pt - pp` 가 아닐까

- `pp == 0` 은 **패턴의 첫 글자까지 비교를 마쳤다**는 뜻. BM은 뒤에서 앞으로 오니까, 이 순간 `pt` 는 **매칭 구간의 시작**을 가리켜.
- 26일차 불변식을 적용하면 `pt - pp = pt - 0 = pt` — **사실 같은 공식**이야! BM은 `pp`가 0이라 생략된 것뿐.

| | 반환 | 그때 pt의 의미 |
|---|---|---|
| BF/KMP | `pt - pp` (pp = m) | 매칭의 **끝 다음** |
| BM | `pt` (pp = 0) | 매칭의 **시작** |

- **`-1` 판정이 다른 이유**: BM은 `while pt < len(txt)` 를 정상 종료하면 무조건 실패야. 성공하면 함수 안에서 `return pt` 로 **즉시 빠져나가**거든. BF/KMP는 루프를 나온 뒤에 판정해야 해서 `pp` 검사가 필요했지.

**세 알고리즘 결과 일치**: 랜덤 3000회 불일치 0회 ✅

---

## ⏱️ PART 5 해설 — 시간 복잡도

### 15. 손으로 세어보기

```
txt = AAAAAAAAAB (n=10), pat = AAB (m=3)
```

**(가) 브루트 포스** — 시작 위치별 비교 횟수: i=0~6은 각각 3번, i=7은 3번(성공)
- ① 시작 위치 개수 = `n - m + 1` = **8**
- ② 각 위치 최대 비교 = **m = 3**
- ③ 총 ≈ **24번** (실측 24번 ✅)
- 👉 **O(m·n)**: "시작 위치 n개 × 각 최대 m번"

**(나) KMP**
- ④ `pt` 최대 증가 = **n**
- ⑤ `pp` 총 감소량 ≤ **n**
- ⑥ 총 비교 ≤ **2n** (실측 17번, n=10)
- 👉 **O(n)**: "두 커서의 총 이동량이 n에 비례"

**(다) 보이어·무어**
- ⑦ `skip['A']` = `3 - 1 - 1` = **1**
- ⑧ 평균 점프 ≈ **m**
- ⑨ 라운드 수 ≈ **n / m**
- ⑩ 총 비교 ≈ **n / m** (실측 10번, n/m = 3.3)
- 👉 **O(n / m)**: "m칸씩 건너뛰니 라운드가 n/m번"

---

### 16. 🔥 O(n/m)을 눈으로 — m을 키우기

**실측 (n = 100,000 고정)**
```
   m  |     BF      |     KMP     |     BM      |    n/m    | BM ÷ (n/m)
    2 |     100,000 |     100,000 |      50,000 |    50,000 |   1.00
    4 |     100,000 |     100,000 |      25,000 |    25,000 |   1.00
    8 |     100,000 |     100,000 |      12,500 |    12,500 |   1.00
   16 |     100,000 |     100,000 |       6,250 |     6,250 |   1.00
   32 |     100,000 |     100,000 |       3,125 |     3,125 |   1.00
   64 |     100,000 |     100,000 |       1,562 |     1,562 |   1.00
```

🔥 **마지막 열이 전부 정확히 1.00.** BM의 비교 횟수 = **n/m 그 자체**야.

- **BF, KMP**: m이 커져도 **100,000 고정** → O(n)
- **BM**: m이 2배 되면 비교가 **절반** → O(n/m)
- m=64면 **BF/KMP보다 64배 적게** 비교해.

> 🔑 이 표 한 장이 O(n)과 O(n/m)의 차이 전부야. **"m이 커질 때 어떻게 되는가"** 를 보면 돼.

---

### 17. n을 키우기

**실측 (m = 8 고정)**
```
      n    |     BF      |     KMP     |     BM      | BM ÷ n
    10,000 |      10,000 |      10,000 |       1,250 |  0.1250
    25,000 |      25,000 |      25,000 |       3,125 |  0.1250
    50,000 |      50,000 |      50,000 |       6,250 |  0.1250
   100,000 |     100,000 |     100,000 |      12,500 |  0.1250
   200,000 |     200,000 |     200,000 |      25,000 |  0.1250
```

- ①②③ 셋 다 **2배** (n에 정비례)
- ④ **차이는 "기울기(비례상수)"** 야. BF/KMP는 계수 1.0, BM은 **1/m = 0.125**.

💡 **Big-O 관점 정리**
- m을 상수로 보면 `O(n/m) = O(n)` 이야. 엄밀히는 같은 클래스지.
- 하지만 **m이 함께 커지는 실제 상황**에서는 결정적 차이가 돼. 그래서 교재도 굳이 `O(n/m)` 이라 쓴 거야.
- 이건 **"상수를 깎는 최적화"** 의 극단적 사례 — 22일차 개선 퀵 정렬(sort3, 삽입 전환)과 같은 계열인데 효과가 훨씬 커.

---

### 18. 🔥 BM의 진짜 최악

**실측** — `txt = 'AAA...A'`, `pat = 'BAA...A'`
```
      n | m   |     BF      |     KMP     |     BM      | BM ÷ n
  10,000 |  10 |      10,000 |      10,000 |      99,910 |   10.0
  20,000 |  10 |      20,000 |      20,000 |     199,910 |   10.0
  10,000 |  50 |      10,000 |      10,000 |     497,550 |   49.8
  10,000 | 100 |      10,000 |      10,000 |     990,100 |   99.0
```

🔥 **BM이 셋 중 가장 느려.** 그리고 `BM ÷ n` 이 정확히 **m과 같아** → **O(n·m)**

**왜**
- 매 라운드 뒤에서 `m-1` 글자가 전부 일치하다가 맨 앞 `B` 에서 불일치
- 그때 `a = skip['A'] = 0` (A의 마지막 출현이 m-1), `b = m - 0 = m` → m칸 전진
- 하지만 그 m칸을 위해 **m번 비교**했어. 글자당 1번씩 본 셈인데, `pt`가 안쪽으로 `m-1`칸 되돌아갔다 나오니 실질 손해가 커.

**교재 319p의 "최악이라도 O(n)"은 정식 BM(배열 2개) 이야기야.** 착한 접미사 규칙이 있어야 이 케이스에서 한 번에 크게 밀 수 있거든. 3번의 각주가 그래서 중요했던 거야.

> 🔑 **간이 버전(나쁜 문자만) = 평균 O(n/m), 최악 O(n·m)**
> **정식 BM(두 규칙) = 평균 O(n/m), 최악 O(n)**

---

### 19. 최종 총정리

| | 비교 방향 | 불일치 시 | 전처리 | 평균 | 최악 | 실무 |
|---|---|---|---|---|---|---|
| 브루트 포스 | 앞→뒤 | `pt` 되돌리고 `pp=0` | 없음 | ① **O(n)** | ② **O(mn)** | 짧은 입력 OK |
| KMP | 앞→뒤 | `pp = skip[pp]` | ③ **O(m)** | ④ **O(n)** | ⑤ **O(n)** | ⑥ 거의 안 씀 |
| 보이어·무어 | ⑦ **뒤→앞** | ⑧ **표 보고 여러 칸 점프** | ⑨ **O(m + 256)** | ⑩ **O(n/m)** | ⑪ **O(n·m)**(간이) / O(n)(정식) | ⑫ **실무 표준** |

**최종 실측 종합**
```
상황                   |         BF |        KMP |         BM | 승자
평범한 영문 텍스트         |    100,000 |    100,000 |     12,500 | BM
알파벳 2종 (DNA 같은)     |      1,169 |        907 |      1,141 | KMP
최악 (반복 문자)          |    100,000 |    100,000 |    999,910 | BF/KMP
```

**최종 질문 답**

**1. 왜 O(n/m)이 O(n)보다 빠른가**
16번 표대로 m이 2배 될 때마다 비교가 절반으로 줄어들어. m=64면 BF/KMP 대비 **64배 적게** 비교해. 텍스트의 **모든 글자를 보지 않기 때문**이야.

**2. "텍스트를 다 안 본다"가 가능한 이유**
BM은 **뒤에서부터** 비교해서, 마지막 글자가 패턴에 없는 글자면 **그 앞 m-1개를 볼 필요가 없다**는 걸 즉시 알 수 있어. BF/KMP는 앞에서부터 순차적으로 가니 모든 글자를 최소 한 번은 봐야 하고.

**3. BM은 스트리밍이 가능할까**
**어려워.** BM은 `pt`가 라운드 안에서 **뒤로 되돌아가**(4번 라운드 2에서 7→6). 파일을 한 번만 읽으며 처리하려면 되돌아간 부분을 버퍼에 갖고 있어야 해. KMP는 `pt`가 절대 안 되돌아가서(27일차 10번) 스트리밍에 유리하지. 교재 319p가 KMP의 유일한 장점으로 이걸 꼽은 이유야.

**4. 언제 뭘 쓰나**
- **일반적인 경우** → `str.find()` (Two-way, C 구현). 실측에서도 가장 빨라.
- **직접 구현해야 하고 알파벳이 다양한 경우** → 보이어·무어
- **입력이 짧거나 일회성** → 브루트 포스 (전처리 없이 바로)
- **스트리밍/한 번만 읽기** → KMP
- **알파벳이 2종 등 극히 적을 때** → KMP 또는 BF (BM은 오히려 손해)

---

## 📌 핵심 3줄 요약

1. **보이어·무어는 "뒤에서부터" 비교한다.** 그래서 패턴에 없는 글자를 만나면 **m-1개 위치를 한꺼번에 배제**하고 점프할 수 있어. 이동량은 `m - k - 1`(k = 그 글자의 패턴 내 마지막 출현), 패턴에 없으면 `m`.
2. **O(n/m)은 "텍스트를 다 안 본다"는 뜻이다.** 한 라운드에 비교 1번 하고 m칸 점프하니 라운드가 n/m번. 실측에서 `BM 비교 횟수 ÷ (n/m) = 1.00` 이 정확히 나왔고, **m이 커질수록 빨라지는 유일한 알고리즘**이야.
3. **이동량 계산은 `max(a, b)` 다.** `a`는 나쁜 문자 규칙, `b = m - pp` 는 최소 보장선. `b`가 없으면 `skip` 값이 0인 패턴(`AAAA`, `EXAMPLE`)에서 무한 루프에 빠져.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 3, 4, 7, 9, 13, 15, 19번)**: 전원 필수
  - **4번 라운드 추적**이 오늘의 기본기. 특히 라운드 2에서 `pt`가 되돌아갔다 점프하는 걸 볼 것
  - **15번 손으로 세기**를 건너뛰면 16~18번 실측이 그냥 숫자로만 보여
- 🟡 **(R-2, 2, 5, 8, 11, 14, 16, 17번)**: 팀 목표선
  - **16번이 오늘의 하이라이트** 🔥 — `BM ÷ (n/m) = 1.00` 을 직접 볼 것. O(n/m)이 뭔지 이 표 하나로 끝나
  - **11번 `max(a, b)`** 는 코드에서 가장 안 읽히는 줄이니 꼭 분해해볼 것
- 🔴 **(6, 10, 12, 18번)**: 도전
  - **18번이 오늘 최고 난도** 🔥🔥 — 교재가 "최악 O(n)"이라 했는데 실측은 O(n·m). 왜 다른지(간이 vs 정식) 설명할 수 있으면 완벽
  - **12번 `pt` 초기화 누락**은 22일차 14번과 짝으로 볼 것
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 19번)

## 🔗 오늘 회수된 개념들

- **26일차 `pt - pp` 불변식** → BM의 `return pt` 는 `pt - 0` (14번)
- **26일차 10번 "근거 없이 밀기" 버그** → 앞쪽 A에 맞추면 안 되는 이유 (2번)
- **27일차 KMP skip 표** → 이름만 같고 완전히 다른 물건 (R-2)
- **27일차 `pt` 되돌아가지 않음** → BM은 되돌아간다 → 스트리밍 불가 (19번 Q3)
- **25일차 도수 정렬 `f[값]`** → `skip[ord(글자)]`, 값을 인덱스로 (9번)
- **25일차 19번 값 범위 폭발** → 한글 넣으면 `IndexError`, dict로 해결 (9번)
- **22일차 14번 "우연히 안전한 코드"** → `pt` 초기화 누락 (12번)
- **22일차 개선 퀵 정렬(상수 깎기)** → O(n/m)은 상수 깎기의 극단 (17번)

---

> 🎉 **07장 문자열 검색 완주!** 브루트 포스 → KMP → 보이어·무어, 3종을 전부 손으로 짰어.
>
> **이번 주 전체 흐름**: "1칸씩 민다" → "본 걸 재활용한다" → "안 본 걸 건너뛴다".
> 같은 문제를 푸는 세 가지 사고방식이었고, **뒤로 갈수록 "미리 계산해두는 표"가 커졌지.**
> 25일차 도수 정렬의 도수 분포표, 27일차 skip 표, 오늘의 이동량 표 — 전부 **전처리로 본 계산을 줄이는** 같은 발상이야.
>
> **다음 진도**: 08장 **리스트** 또는 커리큘럼상 다음 단원. 지금까지는 "배열 안에서" 해결했다면, 이제 **자료구조 자체를 새로 만드는** 이야기로 넘어가.